In [6]:
!pip install -r requirements.txt

  Using cached lightgbm-4.6.0.tar.gz (1.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lightgbm: filename=lightgbm-4.6.0-py3-none-linux_x86_64.whl size=2737775 sha256=1276206c5952165230c1d96d6233f29ecd8f5482c9de217939eb25ccec4679a7
  Stored in directory: /home/ec2-user/.cache/pip/wheels/cb/83/4f/255634f94c01f16556323cbf2c4f92981396f889463f14acd6
Successfully built lightgbm


In [1]:
import boto3
import sagemaker

role = sagemaker.get_execution_role()
print(role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
arn:aws:iam::YOUR_AWS_ACCOUNT_ID:role/SageMakerExecutionRole


In [2]:
s3 = boto3.client("s3")
response = s3.list_buckets()
for bucket in response["Buckets"]:
    print(bucket["Name"])

YOUR_S3_BUCKET


In [3]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model/model.joblib", arcname="model.joblib")

print("model.tar.gz created.")

model.tar.gz created.


In [4]:
BUCKET_NAME = "YOUR_S3_BUCKET"  

s3 = boto3.client("s3")
s3.upload_file("model.tar.gz", BUCKET_NAME, "credit-score/model/model.tar.gz")
print(f"Uploaded to s3://{BUCKET_NAME}/credit-score/model/model.tar.gz")

Uploaded to s3://YOUR_S3_BUCKET/credit-score/model/model.tar.gz


In [12]:
from inference import model_fn

bundle = model_fn("model")

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.4.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.4.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from 

In [13]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

BUCKET = "YOUR_S3_BUCKET"     
MODEL_S3_KEY = "credit-score/model/model.tar.gz"
ENDPOINT_NAME = "credit-score-endpoint"

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"


def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="SageMakerExecutionRole")["Role"]["Arn"]


def deploy_endpoint() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        source_dir=".",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME,
    )
    print(f"Endpoint '{ENDPOINT_NAME}' is live.")
    return predictor


predictor = deploy_endpoint()

Role:      arn:aws:iam::YOUR_AWS_ACCOUNT_ID:role/SageMakerExecutionRole
Model URI: s3://YOUR_S3_BUCKET/credit-score/model/model.tar.gz
Endpoint:  credit-score-endpoint

Deploying endpoint...
-----!Endpoint 'credit-score-endpoint' is live.


In [14]:
import json, boto3

runtime = boto3.client("sagemaker-runtime", region_name="us-east-1")

payload = {
    "instances": [{
        "Month": "January", "Age": 23, "Occupation": "Scientist",
        "Annual_Income": 19114.12, "Monthly_Inhand_Salary": 1824.84,
        "Num_Bank_Accounts": 3, "Num_Credit_Card": 4, "Interest_Rate": 3,
        "Num_of_Loan": 4,
        "Type_of_Loan": "Auto Loan, Credit-Builder Loan, Personal Loan, Home Equity Loan",
        "Delay_from_due_date": 3, "Num_of_Delayed_Payment": 7,
        "Changed_Credit_Limit": 11.27, "Num_Credit_Inquiries": 3,
        "Credit_Mix": "Good", "Outstanding_Debt": 809.98,
        "Credit_Utilization_Ratio": 28.93,
        "Credit_History_Age": "22 Years and 1 Months",
        "Payment_of_Min_Amount": "No", "Total_EMI_per_month": 49.57,
        "Amount_invested_monthly": 149.94,
        "Payment_Behaviour": "High_spent_Medium_value_payments",
        "Monthly_Balance": 312.49
    }]
}

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(payload),
)
result = json.loads(response["Body"].read().decode("utf-8"))
print("Prediction:", result["labels"])
print("Probabilities:", result["probabilities"])

Prediction: ['Good']
Probabilities: [[0.02379648752458521, 0.1150981636929737, 0.861105348782441]]


In [15]:
sm_client = boto3.client("sagemaker", region_name="us-east-1")

print(f"Deleting endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deleted.")
except Exception as e:
    print(f"No endpoint found: {e}")

print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Configuration deleted.")
except Exception as e:
    print(f"No config found: {e}")

print("Cleanup complete.")

Deleting endpoint: credit-score-endpoint...
Endpoint deleted.
Deleting endpoint configuration: credit-score-endpoint...
Configuration deleted.
Cleanup complete.
